# Measuring Areas and Distances


The accuracy of **distance** and **area** calculations and the construction of **buffer zones** depends directly on the **coordinate reference system (CRS)** being used.

When data is in a **geographic coordinate system** such as **WGS 84 (EPSG:4326)**, coordinates are expressed in **degrees**. This is suitable for **displaying features on a map**, but **not for measuring distances or areas directly**.

To get results in **metres** and **square metres**, the data must be reprojected into an appropriate **projected coordinate system** such as **UTM**.

In this section, we will cover:

- why data has to be **reprojected** before you measure anything, and how the **CRS shapes the numbers you get**;
- how to calculate **polygon areas**;
- how to measure **distances between features**.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import osmnx as ox
import geopandas as gpd

# cache OSM responses on disk, so repeating a query does not hit the server again
ox.settings.cache_folder = "../../cache"

### 0.2. Preparing the Data


We start with the district boundary from OpenStreetMap.


In [ ]:
area_name = "Innere Stadt, Vienna, Austria"

admin_border = ox.geocode_to_gdf(area_name)
admin_border.explore(tiles="cartodbpositron")

Its coordinate reference system:


In [ ]:
admin_border.crs

Now the metro station entrances in the same district.


In [ ]:
tags = {"railway": "subway_entrance"}

metro = ox.features_from_place(area_name, tags)

metro.explore(tiles="cartodbpositron")

And their coordinate reference system:


In [ ]:
metro.crs

## 1. Measuring Area

In GeoPandas, every `GeoDataFrame` has a `.geometry` attribute that stores the **geometric objects** of the spatial dataset — points, lines, or polygons.

This attribute provides access to the geometry of each feature and enables various **spatial operations**.

One such operation is area calculation. The `.area` attribute returns the **area of each polygon feature**.

Remember that `.area` returns values **in the units of the coordinate reference system (CRS)**.


### 1.1. Calculating Area — Take 1

We add a column `"area_deg2"` to the `admin_border` GeoDataFrame and store the calculated areas in it.

In [ ]:
admin_border["area_deg2"] = admin_border.geometry.area

The result:


In [ ]:
admin_border[["name", "area_deg2"]]

At this point, you may notice that the area values are **unexpectedly small**. This is because the layer is still in a **geographic coordinate system**.

In geographic coordinate systems such as **WGS 84 (EPSG:4326)**, positions are defined in **degrees of latitude and longitude** rather than in linear units. As a result, area is calculated in **square degrees**.

Such values are difficult to interpret, because **degrees are not a unit of length** and their physical scale **varies with latitude**.

To calculate areas and distances correctly, the data must first be **reprojected** into a coordinate system with metric units, such as **UTM**.

> **Note the `UserWarning`** that may appear when calculating area.
> It indicates that the calculation is being performed on data in a **geographic coordinate system**, and the results may be unreliable. This warning is a reminder to **reproject the data** before taking measurements.


### 1.2. Reprojecting to UTM

Let's reproject the data into the appropriate **UTM zone** so that coordinates are expressed in **metres**.

First, we'll determine the suitable UTM CRS using the `.estimate_utm_crs()` method, then reproject the `admin_border` layer into it.


In [ ]:
utm_crs = admin_border.estimate_utm_crs()

admin_border_utm = admin_border.to_crs(utm_crs)

Confirm that the CRS has changed.


In [ ]:
admin_border_utm.crs

We can see that this CRS uses metres. Let's now recalculate the area.


### 1.3. Calculating Area — Take 2


Now the same for `admin_border_utm`, in a column called `"area_m2"`. The column with the degree-based values is still there, so the two can be compared side by side.

In [ ]:
admin_border_utm["area_m2"] = admin_border_utm.geometry.area

The result:


In [ ]:
admin_border_utm[["name", "area_deg2", "area_m2"]]

The values now look far more realistic, since area is being calculated in **square metres**.

Area values may sometimes be displayed in **scientific notation**, which is a compact way of representing large numbers.

A value of the form `a.bcde+X` means $a.bcde \times 10^{X}$.

For example:
`2.866748e+06` = $2.866748 \times 10^{6}$, or approximately 2.9 million.

This is simply an alternative way of writing a large number.


For convenience, the area can be converted to square kilometres (1 km² = 1,000,000 m²):


In [ ]:
admin_border_utm["area_km2"] = admin_border_utm["area_m2"] / 1_000_000
admin_border_utm[["name", "area_km2"]]

## 2. Measuring Distances

The `.distance()` method calculates the distance between geometric features.
Suppose we want to find out how far each metro station entrance is from the centre of the district.

Since the source data is in a **geographic coordinate system**, we first need to **reproject it into the appropriate UTM zone** so that measurements are returned **in metres**.


### 2.0. Preparing the Data


First, let's reproject the metro layer into the same CRS as the district boundary.


In [ ]:
metro_utm = metro.to_crs(utm_crs)

Now let's define the district centroid, which we will use as the reference point for measuring distances.


In [ ]:
center = admin_border_utm.geometry.centroid.iloc[0]

center_gdf = gpd.GeoDataFrame(geometry=[center], crs=utm_crs)

center_gdf.explore(tiles="cartodbpositron")

The centroid is the geometric centre of a polygon.
We use it here as a proxy for the centre of the district.


### 2.1. Calculating Distances

Now let's calculate the distance from the district centroid to each metro station entrance.


In [ ]:
metro_utm["distance_m"] = metro_utm.geometry.distance(center)

metro_utm[["name", "distance_m"]].sort_values("distance_m").head()

The resulting values represent distances in metres, since the data is now in a projected UTM coordinate system.


> **Euclidean distance**
>
> The `.distance()` method computes **Euclidean distance** between geometries — that is, the **straight-line distance across the plane** between two points.
>
> This is sometimes referred to as **straight-line distance** or **as-the-crow-flies distance**.
>
> In practice, spatial analysis often requires **network distances** — for example, distances along roads or public transport routes.
> We will cover network analysis methods in **Module 4** of the course.
>
> That said, for many analytical tasks **Euclidean distance is a useful approximation**, particularly during exploratory spatial analysis.


## Summary


In this section, we looked at how to **correctly measure areas and distances** using GeoPandas.

We learned:

- why geographic CRS are not suitable for spatial measurements;
- how to calculate areas using `.area`;
- how to measure distances using `.distance()`.

Choosing the right coordinate reference system is one of the most important steps in preparing data for spatial analysis.


> **Before performing spatial measurements, always check:**
>
> - whether the layer is in a **projected CRS**;
> - whether coordinates are expressed in **metres**;
> - whether all layers being used share the **same CRS**.
